<a href="https://colab.research.google.com/github/simionRiccard0/Deep-Learning-Project/blob/main/viral_genomes_identification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

DNA sequences for identifying viral
genomes in human samples (la falda)

In [15]:
!git clone https://github.com/NeuroCSUT/ViraMiner.git

fatal: destination path 'ViraMiner' already exists and is not an empty directory.


In [16]:
import os
os.listdir('ViraMiner/data/DNA_data/')

['fullset_train.csv',
 'exp18_2015_F.csv',
 'exp12_2014_J1.csv',
 'exp3_2013_H4.csv',
 'exp10_2014_G6.csv',
 'exp2_2013_E2_SCC.csv',
 'create_LOO_set.py',
 'exp5_2014_D3.csv',
 'fullset_validation.csv',
 'exp15_2014_P.csv',
 'exp14_2014_O.csv',
 'exp1_2011_N19.csv',
 'exp11_2014_G7.csv',
 'create_dataset.py',
 'fullset_test.csv',
 'exp6_2014_E1.csv',
 'exp9_2014_G5.csv',
 'exp4_2014_B.csv',
 'exp17_2015_F2.csv',
 'exp0_2011_G5.csv',
 'exp7_2014_F1.csv',
 'exp8_2014_G1.csv',
 'exp13_2014_N1.csv',
 'exp16_2014_R1.csv']

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

In [18]:
import pandas as pd

train = pd.read_csv(
    'ViraMiner/data/DNA_data/fullset_train.csv',
    header=None
)

val = pd.read_csv(
    'ViraMiner/data/DNA_data/fullset_validation.csv',
    header=None
)

test = pd.read_csv(
    'ViraMiner/data/DNA_data/fullset_test.csv',
    header=None
)

print("Train shape:", train.shape)
print(train.head())

Train shape: (211239, 3)
                                                 0  \
0  seq1006848_2014_P_Lagheden_Matti-HPV-vacc-CIN31   
1                     seq4116690_2014_G6_Hultin_MS   
2            seq360982_2014_F1_PARAFFINBLANKBLOCKS   
3             seq1135576_2011_N19_VIRASKINFAPMISEQ   
4                        seq277_2014_G6_Hultin_MS8   

                                                   1  2  
0  CAAGCCAAGATTTTCTCGCGTCACACTACTCATGACCATTGTATTA...  0  
1  AACGAAGCACGGGCCGAGAGATTGAGGAACCAAGGTCCAGCTCTAG...  0  
2  TAGTGGGTGAGGTTTCTATTTCCATAATGATCTCGCCTCAATTACT...  0  
3  ATATGACCATTCTTGCAAGGTAACACAGGTACATTTTCACAAAGTG...  0  
4  GGTCTTAAAACAACAGAAATTTTTTCCATCACAGTTGCAGAAATTA...  0  


In [19]:
print("Sequence length:", len(train[1][0]))

print("\nClass balance:")
print(train[2].value_counts())

Sequence length: 300

Class balance:
2
0    206773
1      4466
Name: count, dtype: int64


DNA sequences encoding:

In [21]:
import torch
from torch.utils.data import Dataset
#DNA one-hot encoding
mapping = {
    "A": [1,0,0,0],
    "C": [0,1,0,0],
    "G": [0,0,1,0],
    "T": [0,0,0,1],
    "N": [0,0,0,0]
}

MAX_LEN = 300

def encode_sequence(seq):

    seq = seq[:MAX_LEN]

    encoded = [
        mapping.get(base, [0,0,0,0])
        for base in seq
    ]

    while len(encoded) < MAX_LEN:
        encoded.append([0,0,0,0])

    return encoded


class DNADataset(Dataset):

    def __init__(self, dataframe):

        self.sequences = dataframe[1].values
        self.labels = dataframe[2].values

    def __len__(self):

        return len(self.sequences)

    def __getitem__(self, idx):

        seq = self.sequences[idx]
        label = self.labels[idx]

        x = torch.tensor(
            encode_sequence(seq),
            dtype=torch.float32
        )

        y = torch.tensor(
            label,
            dtype=torch.float32
        )

        return x, y

Preparing the dataset (DataLoader):

In [22]:
from torch.utils.data import DataLoader

train_dataset = DNADataset(train)
val_dataset   = DNADataset(val)
test_dataset  = DNADataset(test)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

X_batch, y_batch = next(iter(train_loader))

print(X_batch.shape)
print(y_batch.shape)

torch.Size([128, 300, 4])
torch.Size([128])


CNN Branches

In [23]:
import torch.nn as nn
class PatternBranch(nn.Module):

    def __init__(self):
        super().__init__()


        self.conv6  = nn.Conv1d(4, 512, 6, padding=3)
        self.conv12 = nn.Conv1d(4, 512, 12, padding=6)
        self.conv18 = nn.Conv1d(4, 512, 18, padding=9)

        self.bn = nn.BatchNorm1d(512*3)

        self.pool = nn.AdaptiveMaxPool1d(1)

        self.fc = nn.Sequential(
            nn.Linear(512*3, 512),
            nn.ReLU(),
            nn.Dropout(0.6)
        )

    def forward(self, x):

        x = x.permute(0,2,1)

        x6  = torch.relu(self.conv6(x))
        x12 = torch.relu(self.conv12(x))
        x18 = torch.relu(self.conv18(x))

        x = torch.cat([x6, x12, x18], dim=1)

        x = self.bn(x)

        x = self.pool(x)

        x = x.squeeze(-1)

        x = self.fc(x)

        return x

In [24]:
class FrequencyBranch(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv6  = nn.Conv1d(4, 256, 6, padding=3)
        self.conv12 = nn.Conv1d(4, 256, 12, padding=6)
        self.conv18 = nn.Conv1d(4, 256, 18, padding=9)

        self.bn = nn.BatchNorm1d(256*3)

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Sequential(
            nn.Linear(256*3, 512),
            nn.ReLU(),
            nn.Dropout(0.6)
        )

    def forward(self, x):

        x = x.permute(0,2,1)

        x6  = torch.relu(self.conv6(x))
        x12 = torch.relu(self.conv12(x))
        x18 = torch.relu(self.conv18(x))

        x = torch.cat([x6, x12, x18], dim=1)

        x = self.bn(x)

        x = self.pool(x)

        x = x.squeeze(-1)

        x = self.fc(x)

        return x

Transformer branch

In [25]:
class TransformerBranch(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Linear(4, 128)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=8,
            dim_feedforward=256,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):

        x = self.embedding(x)

        x = self.transformer(x)

        x = x.permute(0, 2, 1)

        x = self.pool(x)

        return x.squeeze(-1)

ViraExplorer Model (CNN Pattern branch + CNN Frequency branch + Transformer)

In [26]:
class ViraExplorer(nn.Module):

    def __init__(self):

        super().__init__()

        self.pattern = PatternBranch()

        self.frequency = FrequencyBranch()

        self.transformer = TransformerBranch()

        self.fc = nn.Sequential(
          nn.Linear(1152, 256),
          nn.ReLU(),
          nn.Dropout(0.6),
          nn.Linear(256, 1)
      )

    def forward(self, x):

        p = self.pattern(x)

        f = self.frequency(x)

        t = self.transformer(x)

        x = torch.cat([p, f, t], dim=1)

        return self.fc(x)

Focal Loss

In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):

    def __init__(self, alpha=0.25, gamma=2):
        super().__init__()

        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):

        targets = targets.unsqueeze(1)

        BCE_loss = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction='none'
        )

        pt = torch.exp(-BCE_loss)

        focal_loss = self.alpha * (1 - pt) ** self.gamma * BCE_loss

        return focal_loss.mean()

Training Loop

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model = ViraExplorer().to(device)

criterion = FocalLoss(
    alpha=0.25,
    gamma=2
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=5e-5,
    weight_decay=1e-5
)

scaler = torch.cuda.amp.GradScaler()

from sklearn.metrics import roc_auc_score

best_auc = 0
patience = 8
counter = 0

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    patience=8,
    factor=0.5
)

EPOCHS = 40

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}, "
        f"Loss: {avg_loss:.4f}"
    )

    # Validation
    model.eval()

    preds = []
    targets = []

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)

            outputs = model(X_batch)

            probs = torch.sigmoid(outputs)

            preds.extend(
                probs.cpu().numpy().flatten()
            )

            targets.extend(
                y_batch.cpu().numpy()
            )

    auc = roc_auc_score(
        targets,
        preds
    )

    print(
        f"Validation AUC: {auc:.4f}"
    )

    scheduler.step(auc)

    if auc > best_auc:

        best_auc = auc
        counter = 0

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

    else:

        counter += 1

        if counter >= patience:

            print("Early stopping triggered")
            break

CUDA available: True
GPU: Tesla T4


/tmp/ipykernel_621/1274085502.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 1, Loss: 0.0081
Validation AUC: 0.7761
Epoch 2, Loss: 0.0066
Validation AUC: 0.8352
Epoch 3, Loss: 0.0062
Validation AUC: 0.8441
Epoch 4, Loss: 0.0061
Validation AUC: 0.8490
Epoch 5, Loss: 0.0059
Validation AUC: 0.8551
Epoch 6, Loss: 0.0058
Validation AUC: 0.8634
Epoch 7, Loss: 0.0057
Validation AUC: 0.8689
Epoch 8, Loss: 0.0055
Validation AUC: 0.8699
Epoch 9, Loss: 0.0054
Validation AUC: 0.8759
Epoch 10, Loss: 0.0053
Validation AUC: 0.8774
Epoch 11, Loss: 0.0052
Validation AUC: 0.8790
Epoch 12, Loss: 0.0051
Validation AUC: 0.8853
Epoch 13, Loss: 0.0050
Validation AUC: 0.8877
Epoch 14, Loss: 0.0049
Validation AUC: 0.8883
Epoch 15, Loss: 0.0048
Validation AUC: 0.8889
Epoch 16, Loss: 0.0047
Validation AUC: 0.8951
Epoch 17, Loss: 0.0046
Validation AUC: 0.8950
Epoch 18, Loss: 0.0046
Validation AUC: 0.8939
Epoch 19, Loss: 0.0045
Validation AUC: 0.8959
Epoch 20, Loss: 0.0044
Validation AUC: 0.8984
Epoch 21, Loss: 0.0044
Validation AUC: 0.9002
Epoch 22, Loss: 0.0043
Validation AUC: 0.90